# Load and preprocess a text dataset, then train Word2Vec models using CBOW (sg=0) and Skip-Gram (sg=1), and display vocabulary size along with word vectors for at least five words.Find the top 5 similar words for a given term using both models and compare their results.

In [1]:
!pip install gensim -q

import re
from gensim.models import Word2Vec


# Dataset


In [3]:
raw_text = """
The cat sat on the mat. The cat is a small animal.
Dogs are loyal animals. Dogs love to play and run.
The king rules the kingdom. The queen rules beside the king.
A man walked into the forest. The woman followed the man.
The lion is the king of the jungle. The tiger is a fierce animal.
Birds fly high in the sky. Fish swim deep in the ocean.
The sun rises in the east and sets in the west.
Trees grow tall in the forest. Flowers bloom in spring.
The dog chased the cat around the house all day long.
Animals live in different habitats across the world.
The boy played with the dog in the park near the river.
The girl read a book about wild animals and their habitats.
Rivers flow from mountains to the ocean below.
The forest is home to many animals including deer and foxes.
Cats and dogs are the most popular pets in the world.
The lion hunts prey in the savanna grasslands of Africa.
Tigers are endangered animals that live in Asian forests.
Eagles are powerful birds that soar above mountain peaks.
Dolphins are intelligent animals that live in the ocean.
The wise old man told stories about the ancient kingdom.
A brave woman led her people through the dark forest safely.
Children love to learn about animals and nature around them.
The queen was as powerful as the king in the royal court.
"""

print(f"loaded {len(raw_text)} characters")

loaded 1300 characters


##  Preprocessing

In [4]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    sentences = []
    for line in text.strip().split('\n'):
        words = line.strip().split()
        if len(words) > 1:
            sentences.append(words)
    return sentences

sentences = preprocess(raw_text)
all_words = [w for sent in sentences for w in sent]

print(f"sentences : {len(sentences)}")
print(f"tokens    : {len(all_words)}")
print(f"unique    : {len(set(all_words))}")
print()
print("first 3 sentences:")
for s in sentences[:3]:
    print(' ', s)

sentences : 23
tokens    : 243
unique    : 130

first 3 sentences:
  ['the', 'cat', 'sat', 'on', 'the', 'mat', 'the', 'cat', 'is', 'a', 'small', 'animal']
  ['dogs', 'are', 'loyal', 'animals', 'dogs', 'love', 'to', 'play', 'and', 'run']
  ['the', 'king', 'rules', 'the', 'kingdom', 'the', 'queen', 'rules', 'beside', 'the', 'king']


## Training the models

- `vector_size=50` — embedding dimensions
- `window=3` — context window on each side
- `min_count=1` — keep all words (corpus too small to filter)
- `sg=0` → CBOW, `sg=1` → Skip-Gram
- `epochs=200` — more iterations since dataset is tiny
- `seed=42` — reproducibility

In [5]:
# CBOW
cbow = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=0,
    epochs=200,
    seed=42
)

# Skip-Gram
skipgram = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=200,
    seed=42
)

## Vocab Size

In [7]:
vocab = list(cbow.wv.key_to_index.keys())
print(f"vocab size: {len(vocab)} words")
print()
print(sorted(vocab))

vocab size: 130 words

['a', 'about', 'above', 'across', 'africa', 'all', 'ancient', 'and', 'animal', 'animals', 'are', 'around', 'as', 'asian', 'below', 'beside', 'birds', 'bloom', 'book', 'boy', 'brave', 'cat', 'cats', 'chased', 'children', 'court', 'dark', 'day', 'deep', 'deer', 'different', 'dog', 'dogs', 'dolphins', 'eagles', 'east', 'endangered', 'fierce', 'fish', 'flow', 'flowers', 'fly', 'followed', 'forest', 'forests', 'foxes', 'from', 'girl', 'grasslands', 'grow', 'habitats', 'her', 'high', 'home', 'house', 'hunts', 'in', 'including', 'intelligent', 'into', 'is', 'jungle', 'king', 'kingdom', 'learn', 'led', 'lion', 'live', 'long', 'love', 'loyal', 'man', 'many', 'mat', 'most', 'mountain', 'mountains', 'nature', 'near', 'ocean', 'of', 'old', 'on', 'park', 'peaks', 'people', 'pets', 'play', 'played', 'popular', 'powerful', 'prey', 'queen', 'read', 'rises', 'river', 'rivers', 'royal', 'rules', 'run', 'safely', 'sat', 'savanna', 'sets', 'sky', 'small', 'soar', 'spring', 'stories'

##  Word Vectors for 5 Words



In [8]:
words_to_check = ['cat', 'dog', 'king', 'woman', 'forest']

print(f"{'word':<10}  {'model':<12}  first 10 vector values")
print("-" * 70)

for word in words_to_check:
    for label, model in [("CBOW", cbow), ("Skip-Gram", skipgram)]:
        vec = model.wv[word]
        vals = ', '.join([f'{v:.3f}' for v in vec[:10]])
        print(f"{word:<10}  {label:<12}  [{vals}...]")
    print()

word        model         first 10 vector values
----------------------------------------------------------------------
cat         CBOW          [-0.057, -0.038, -0.230, 0.314, -0.197, 0.303, -0.189, 0.405, -0.453, 0.012...]
cat         Skip-Gram     [-0.140, -0.067, -0.143, 0.362, -0.082, 0.134, -0.112, 0.185, -0.271, 0.097...]

dog         CBOW          [-0.024, -0.037, -0.174, 0.224, -0.196, 0.234, -0.181, 0.377, -0.355, -0.030...]
dog         Skip-Gram     [-0.085, -0.087, -0.113, 0.239, -0.185, 0.126, -0.192, 0.289, -0.233, 0.009...]

king        CBOW          [-0.034, -0.060, -0.187, 0.213, -0.150, 0.251, -0.178, 0.355, -0.379, -0.013...]
king        Skip-Gram     [-0.128, -0.123, -0.154, 0.160, -0.080, 0.148, -0.192, 0.255, -0.293, -0.050...]

woman       CBOW          [-0.011, -0.034, -0.200, 0.260, -0.167, 0.260, -0.199, 0.368, -0.398, -0.020...]
woman       Skip-Gram     [-0.082, -0.078, -0.204, 0.253, 0.001, 0.115, -0.217, 0.272, -0.314, -0.074...]

forest      CBOW        

## Top 5 Similar Words (Comparison)



In [9]:
query_words = ['cat', 'king', 'forest', 'animal', 'man']

for qw in query_words:
    print(f"\nQuery: '{qw}'")
    print(f"  {'#':<4}  {'CBOW':<30}  {'Skip-Gram'}")
    print(f"  {'--':<4}  {'-'*28:<30}  {'-'*28}")

    cbow_sim = cbow.wv.most_similar(qw, topn=5)
    sg_sim   = skipgram.wv.most_similar(qw, topn=5)

    for i, ((cw, cs), (sw, ss)) in enumerate(zip(cbow_sim, sg_sim), 1):
        cb = f"{cw:<15} {cs:.4f}"
        sk = f"{sw:<15} {ss:.4f}"
        print(f"  {i:<4}  {cb:<30}  {sk}")


Query: 'cat'
  #     CBOW                            Skip-Gram
  --    ----------------------------    ----------------------------
  1     day             0.9965          mat             0.9867
  2     mat             0.9962          on              0.9859
  3     the             0.9959          sat             0.9836
  4     a               0.9959          small           0.9783
  5     on              0.9959          house           0.9601

Query: 'king'
  #     CBOW                            Skip-Gram
  --    ----------------------------    ----------------------------
  1     the             0.9962          beside          0.9820
  2     is              0.9954          rules           0.9818
  3     a               0.9952          queen           0.9813
  4     of              0.9949          the             0.9693
  5     dog             0.9949          was             0.9664

Query: 'forest'
  #     CBOW                            Skip-Gram
  --    ----------------------------

## Conclusion

- CBOW is faster and works well for frequent words
- Skip-Gram handles rare words better and gives richer vectors on large corpora
- both models encode semantic similarity geometrically
- with a proper large corpus the famous analogy king - man + woman ≈ queen would work